In [8]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

# ==========================================
# 1. DATA STRUCTURES
# ==========================================
class Stack:
    def __init__(self):
        self.items = []
    def push(self, item):
        self.items.append(item)
    def pop(self):
        return self.items.pop() if not self.is_empty() else None
    def is_empty(self):
        return len(self.items) == 0

class BinaryTree:
    def __init__(self, rootObj):
        self.key = rootObj
        self.leftChild = None
        self.rightChild = None
        self.original_id = id(self)

    def insertLeft(self, newNode):
        t = BinaryTree(newNode)
        if self.leftChild == None:
            self.leftChild = t
        else:
            t.leftChild = self.leftChild
            self.leftChild = t

    def insertRight(self, newNode):
        t = BinaryTree(newNode)
        if self.rightChild == None:
            self.rightChild = t
        else:
            t.rightChild = self.rightChild
            self.rightChild = t

    def getRightChild(self):
        return self.rightChild

    def getLeftChild(self):
        return self.leftChild

    def setRootVal(self, obj):
        self.key = obj

    def getRootVal(self):
        return self.key

def copy_tree(node):
    if node is None:
        return None
    new_node = BinaryTree(node.key)
    new_node.original_id = node.original_id
    new_node.leftChild = copy_tree(node.leftChild)
    new_node.rightChild = copy_tree(node.rightChild)
    return new_node

# ==========================================
# 2. PARSE ALGORITHM & STATE TRACKER
# ==========================================
def buildParseTree_with_states(fpexp):
    fplist = fpexp.split()
    pStack = Stack()
    eTree = BinaryTree('')
    pStack.push(eTree)
    currentTree = eTree
    
    states = []
    step_count = 0
    
    def save_state(token, action):
        nonlocal step_count
        stack_snapshot = [str(n.key) if n.key != '' else '?' for n in pStack.items]
        states.append({
            'step': step_count,
            'token': token,
            'action': action,
            'current_node_id': currentTree.original_id if currentTree else None,
            'stack': stack_snapshot,
            'tree': copy_tree(eTree)
        })
        step_count += 1

    save_state("-", "Initialize")

    for i in fplist:
        if i == '(':
            currentTree.insertLeft('')
            pStack.push(currentTree)
            currentTree = currentTree.getLeftChild()
            save_state(i, "RULE 1: insertLeft, push")
        elif i in ['+', '-', '*', '/']:
            currentTree.setRootVal(i)
            currentTree.insertRight('')
            pStack.push(currentTree)
            currentTree = currentTree.getRightChild()
            save_state(i, "RULE 2: setRoot, insertRight, push")
        elif i == ')':
            currentTree = pStack.pop()
            save_state(i, "RULE 4: pop")
        elif i not in ['+', '-', '*', '/', ')']:
            try:
                currentTree.setRootVal(int(i))
                parent = pStack.pop()
                currentTree = parent
                save_state(i, "RULE 3: setRoot, pop")
            except ValueError:
                print(f"Error: Token '{i}' tidak valid.")
                
    return states

# ==========================================
# 3. DRAWING LOGIC
# ==========================================
def get_coords(node, x, y, dx, dy, pos_dict):
    if node is not None:
        pos_dict[node.original_id] = (x, y)
        if node.leftChild:
            get_coords(node.leftChild, x - dx, y - dy, dx / 1.8, dy, pos_dict)
        if node.rightChild:
            get_coords(node.rightChild, x + dx, y - dy, dx / 1.8, dy, pos_dict)
    return pos_dict

def draw_tree(ax, node, current_id, x, y, dx, dy):
    coords = get_coords(node, x, y, dx, dy, {})
    
    def draw_edges(n):
        if n:
            x1, y1 = coords[n.original_id]
            if n.leftChild:
                x2, y2 = coords[n.leftChild.original_id]
                ax.plot([x1, x2], [y1, y2], 'k-', zorder=1)
                draw_edges(n.leftChild)
            if n.rightChild:
                x2, y2 = coords[n.rightChild.original_id]
                ax.plot([x1, x2], [y1, y2], 'k-', zorder=1)
                draw_edges(n.rightChild)
    draw_edges(node)
    
    for nid, (nx, ny) in coords.items():
        color = '#4CAF50' if nid == current_id else '#2196F3'
        
        def find_val(n, target_id):
            if not n: return None
            if n.original_id == target_id: return n.key
            return find_val(n.leftChild, target_id) or find_val(n.rightChild, target_id)
            
        val = find_val(node, nid)
        display_val = str(val) if val != '' else '?'
        
        circle = plt.Circle((nx, ny), 0.15, color=color, zorder=2)
        ax.add_patch(circle)
        ax.text(nx, ny, display_val, color='white', ha='center', va='center', 
                fontsize=12, fontweight='bold', zorder=3)

# ==========================================
# 4. ANIMATION EXPORT ENGINE (UNTUK JUPYTER BOOK)
# ==========================================
def generate_interactive_tree(expression):
    states = buildParseTree_with_states(expression)
    
    fig, (ax_tree, ax_trace) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [2, 1]})
    fig.suptitle(f"Parsing: {expression}", fontsize=16, fontweight='bold')
    
    def update(frame):
        state = states[frame]
        
        ax_tree.clear()
        ax_tree.set_title(f"Tree Visualization - Step {state['step']}", fontsize=12)
        ax_tree.axis('off')
        ax_tree.set_xlim(-3, 3)
        ax_tree.set_ylim(-4, 1)
        
        if state['tree']:
            draw_tree(ax_tree, state['tree'], state['current_node_id'], 0, 0, 1.5, 1)
            
        ax_trace.clear()
        ax_trace.axis('off')
        
        trace_text = (
            f"--- PROGRAM TRACE ---\n\n"
            f"Step      : {state['step']} / {len(states)-1}\n"
            f"Token     : [ {state['token']} ]\n\n"
            f"Action    : \n{state['action']}\n\n"
            f"Stack     : \n{state['stack']}\n\n"
            f"---------------------\n"
            f"* Hijau = Current Node\n"
            f"* Biru  = Parent/Child"
        )
        
        ax_trace.text(0.05, 0.9, trace_text, fontsize=12, va='top', family='monospace', 
                      bbox=dict(facecolor='#f4f4f4', edgecolor='black', boxstyle='round,pad=1'))
        
    ani = animation.FuncAnimation(fig, update, frames=len(states), interval=1000, repeat=False)
    plt.close()
    
    return display(HTML(ani.to_jshtml()))

# ==========================================
# 5. EXECUTE (DENGAN CUSTOM INPUT)
# ==========================================
print("=== PARSE TREE VISUALIZER ===")
print("Syarat: Pisahkan setiap angka dan kurung dengan SPASI.")
print("Contoh: ( ( 30 + 80 ) * 39 )")
print("=============================")

# Meminta input dari user di JupyterLab
ekspresi_custom = input("Masukkan ekspresi matematikamu: ")

# Jika user langsung menekan Enter tanpa mengisi, gunakan Test Case 1
if ekspresi_custom.strip() == "":
    print("Input kosong. Menjalankan default: ( 3 + ( 4 * 5 ) )")
    ekspresi_custom = "( 3 + ( 4 * 5 ) )"

# Eksekusi dan tampilkan animasinya
generate_interactive_tree(ekspresi_custom)

=== PARSE TREE VISUALIZER ===
Syarat: Pisahkan setiap angka dan kurung dengan SPASI.
Contoh: ( ( 30 + 80 ) * 39 )


Masukkan ekspresi matematikamu:  ( 10 + ( 5 * 2 ) )
